In [ ]:
import scanpy as sc
import pandas as pd
import plotnine as gg
import glob
import matplotlib.pyplot as plt

from essential.utils import PLOTNINE_DEFAULT_THEME_2

plt.rcParams["svg.fonttype"] = "none"

In [ ]:
MODEL_NAMES = {
    "gaussian_causal": "Gaussian causal",
    "nb_causal": "NB causal",
    "nb_linear": "NB linear",
    "mean_baseline": "training mean",
    "nb_causal_ds": "NB causal (deepsets)",
    "nb_causal_rollout": "NB causal (rollout training)",
    "nb_causal_residual": "NB causal (residual)",
}

### Pearson R boxplots

In [ ]:
files = glob.glob(
    "/workspace/experiments/06052026_cellbox_noise/results/*/metrics/perturbation_metrics.csv"
)
df = pd.concat([pd.read_csv(f).assign(model_=f.split("/")[-3]) for f in files]).assign(
    model=lambda x: x["model_"].map(MODEL_NAMES)
)
df["model_"].unique()

In [ ]:
df.groupby("model_")["lfc_pearson_r"].mean()

In [ ]:
models = [
    "Gaussian causal",
    "NB causal",
    "NB linear",
    "training mean",
    # "nb_causal_ds": "NB causal (deepsets)",
    # "nb_causal_rollout": "NB causal (rollout training)",
]

fig = (
    gg.ggplot(
        df.loc[lambda x: x["model"].isin(models)],
        gg.aes(x="model", y="lfc_pearson_r", fill="model"),
    )
    + gg.geom_boxplot(outlier_alpha=0, width=0.3)
    + gg.geom_jitter(fill="black", width=0.1, size=0.8, stroke=0.0)
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(
        figure_size=(3.0, 2),
        # legend_position="none",
        panel_grid_major_y=gg.element_line(color="grey", size=0.1),
        axis_text_x=gg.element_text(angle=45, ha="right"),
    )
    + gg.labs(
        x="",
        y="Pearson r",
    )
)
fig.save("no_rollout_pearson_r.svg")
fig

In [ ]:
df.query("model == 'training mean'").sort_values("lfc_pearson_r", ascending=False).head(
    5
)

In [ ]:
models = [
    # "Gaussian causal",
    "NB causal",
    # "NB linear",
    # "training mean",
    "NB causal (deepsets)",
    "NB causal (rollout training)",
    "NB causal (residual)",
]

In [ ]:
fig = (
    gg.ggplot(
        df.loc[lambda x: x["model"].isin(models)],
        gg.aes(x="model", y="lfc_pearson_r", fill="model"),
    )
    + gg.geom_boxplot(outlier_alpha=0, width=0.3)
    + gg.geom_jitter(fill="black", width=0.1, size=0.8, stroke=0.0)
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(
        figure_size=(2.5, 2),
        # legend_position="none",
        panel_grid_major_y=gg.element_line(color="grey", size=0.1),
        axis_text_x=gg.element_text(angle=45, ha="right"),
    )
    + gg.labs(
        x="",
        y="Pearson r",
    )
)
fig.save("pearson_r_ds_rollouts.svg")
fig

### LFC scatters for a singled-out gene

In [ ]:
train_lfc_pred = pd.read_csv(
    "/workspace/experiments/06052026_cellbox_noise/results/mean_baseline/metrics/lfc_pred.csv",
    index_col=0,
)
lfc_gt = pd.read_csv(
    "/workspace/experiments/06052026_cellbox_noise/results/mean_baseline/metrics/lfc_gt.csv",
    index_col=0,
)

In [ ]:
gene = "aaer"
gene_lfc_pred = train_lfc_pred.loc[gene]
gene_lfc_gt = lfc_gt.loc[gene]
plot_df = pd.DataFrame({"predicted LFC": gene_lfc_pred, "GT LFC": gene_lfc_gt})

fig = (
    gg.ggplot(plot_df, gg.aes(x="GT LFC", y="predicted LFC"))
    + gg.geom_point(size=1.0, stroke=0.0)
    + PLOTNINE_DEFAULT_THEME_2
)
fig.save(f"{gene}_train_avg_lfc_scatter.png", dpi=500)
fig

In [ ]:
def compute_cv(adata_path):
    adata = sc.read_h5ad(adata_path)
    std = adata.X.std(0)
    mean = adata.X.mean(0)
    cv = std / (mean + 1e-8)
    return cv


nb_causal_cv = compute_cv(
    "/workspace/experiments/06052026_cellbox_noise/results/nb_causal/adata_pred.h5ad"
)
nb_linear_cv = compute_cv(
    "/workspace/experiments/06052026_cellbox_noise/results/nb_linear/adata_pred.h5ad"
)
nb_gaussian_causal_cv = compute_cv(
    "/workspace/experiments/06052026_cellbox_noise/results/gaussian_causal/adata_pred.h5ad"
)

nb_causal_cv = pd.DataFrame({"cv": nb_causal_cv, "model": "NB causal"})
nb_linear_cv = pd.DataFrame({"cv": nb_linear_cv, "model": "NB linear"})
nb_gaussian_causal_cv = pd.DataFrame(
    {"cv": nb_gaussian_causal_cv, "model": "Gaussian causal"}
)
cv_df = pd.concat([nb_causal_cv, nb_linear_cv, nb_gaussian_causal_cv])
cv_df

In [ ]:
fig = (
    gg.ggplot(cv_df, gg.aes(x="model", y="cv", fill="model"))
    + gg.geom_boxplot(outlier_alpha=0, width=0.3)
    + gg.scale_y_log10()
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(
        panel_grid_major_y=gg.element_line(color="grey", size=0.3),
        panel_grid_minor_y=gg.element_line(color="grey", size=0.1),
        axis_text_x=gg.element_text(angle=45, ha="right"),
    )
    + gg.labs(
        x="",
        y="CV (log scale)",
    )
)
fig.save("no_rollout_cv.svg")

### Rollouts, DS

In [ ]:
train_lfc_pred = pd.read_csv(
    "/workspace/experiments/06052026_cellbox_noise/results/mean_baseline/metrics/lfc_pred.csv",
    index_col=0,
)
lfc_gt = pd.read_csv(
    "/workspace/experiments/06052026_cellbox_noise/results/mean_baseline/metrics/lfc_gt.csv",
    index_col=0,
)

### LFCs

In [ ]:
lfc_pred = pd.read_csv(
    "/workspace/experiments/06052026_cellbox_noise/results/nb_causal_residual/metrics/lfc_pred.csv",
    index_col=0,
)
lfc_gt = pd.read_csv(
    "/workspace/experiments/06052026_cellbox_noise/results/nb_causal_residual/metrics/lfc_gt.csv",
    index_col=0,
)

In [ ]:
gene = "laci"
gene_lfc_pred = lfc_pred.loc[gene]
gene_lfc_gt = lfc_gt.loc[gene]
plot_df = pd.DataFrame({"predicted LFC": gene_lfc_pred, "GT LFC": gene_lfc_gt})

fig = (
    gg.ggplot(plot_df, gg.aes(x="GT LFC", y="predicted LFC"))
    + gg.geom_point(size=1.0, stroke=0.0)
    + PLOTNINE_DEFAULT_THEME_2
)
fig

In [ ]:
df.query("model == 'NB causal (residual)'").sort_values(
    "lfc_pearson_r", ascending=False
).head(5)

In [ ]:
gene = "laci"
gene_lfc_pred = train_lfc_pred.loc[gene]
gene_lfc_gt = lfc_gt.loc[gene]
plot_df = pd.DataFrame({"predicted LFC": gene_lfc_pred, "GT LFC": gene_lfc_gt})

fig = (
    gg.ggplot(plot_df, gg.aes(x="GT LFC", y="predicted LFC"))
    + gg.geom_point(size=1.0, stroke=0.0)
    + PLOTNINE_DEFAULT_THEME_2
)
fig

### Train on evaluation

In [ ]:
files = glob.glob(
    "/workspace/experiments/06052026_cellbox_noise/results/*/metrics_train/perturbation_metrics.csv"
)
df = pd.concat([pd.read_csv(f).assign(model_=f.split("/")[-3]) for f in files]).assign(
    model=lambda x: x["model_"].map(MODEL_NAMES)
)
df["model_"].unique()

In [ ]:
df.groupby("model")["lfc_pearson_r"].mean()